In [2]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic
import json

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [9]:
# Helper functions
def add_user_message(messages, text) -> list[str]:
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text) -> list[str]:
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]) -> str:
    params = {
        "model": model,
        "max_tokens": 1500,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [16]:

# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Solution Criteria, use this to evaluate the solution:
<solutionCriteria>
{test_case["solution_criteria"]}
</solutionCriteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [5]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [13]:
# helper functions for running the test cases through claude

from statistics import mean

TASK_PROMPT = """\
Please solve the following task:

{task}

* Respond only with Python, JSON, or a regex
* Do not add comments, commantary, or explanation
"""


def run_prompt(test_case) -> str:
    """Merges the prompt and test case input"""

    prompt = TASK_PROMPT.format(task=test_case["task"])
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code") #code is a bit more generic than we've seen before
    return chat(messages, stop_sequences=["```"])


def run_test_case(test_case) -> float:
    """Calls run_prompt and grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    return {
        "output": output,
        "test_case": test_case,
        "model_score": model_score,
        "syntax_score": syntax_score,
        "reasoning": reasoning
    }


def run_eval(dataset):
    """Loads the dataset and runs run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_model_score = mean([result["model_score"] for result in results])
    average_syntax_score = mean([result["syntax_score"] for result in results])

    print(f"Average Model Score: {average_model_score}" )
    print(f"Average Syntax Score: {average_syntax_score}" )

    return results

In [17]:
import json

with open("dataset.json", "r") as file:
    dataset = json.load(file)

results = run_eval(dataset)

Average Model Score: 6.666666666666667
Average Syntax Score: 6.666666666666667


In [18]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport re\nimport json\n\ndef parse_cloudwatch_log(log_entry):\n    pattern = r'\\[([^\\]]+)\\]\\s+\\[([^\\]]+)\\]\\s+(.*)'\n    match = re.match(pattern, log_entry)\n    if match:\n        return {\n            \"timestamp\": match.group(1),\n            \"level\": match.group(2),\n            \"message\": match.group(3)\n        }\n    return None\n\nlog = \"[2024-01-15T10:30:45Z] [ERROR] Database connection failed\"\nresult = parse_cloudwatch_log(log)\nprint(json.dumps(result, indent=2))\n",
    "test_case": {
      "task": "Parse an AWS CloudWatch log entry and extract the timestamp, log level, and message. The log format is: '[TIMESTAMP] [LEVEL] MESSAGE'",
      "format": "regex",
      "solution_criteria": "The regex should correctly capture three groups: timestamp (ISO 8601 format), log level (DEBUG, INFO, WARN, ERROR), and the message text. It should handle multi-word messages."
    },
    "model_score": 7,
    "syntax_score": 0,
    "reasoning": "The sol